<a href="https://colab.research.google.com/github/MananAslamDev/ML-Stuff/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The Rule (Plain Words):
A page gets flagged for review if it is older than 180 days (staleness) AND it averages more than 50 search impressions per day (volume).

The Reason Code & Action:

Reason Code: STALE_HIGH_VOLUME

Action Label: REFRESH_REVIEW

The Score (Ranking):
To prioritize the editors' time, the score is simply the historical avg_impressions. The pages with the highest baseline visibility (and therefore the most traffic at risk of decaying due to staleness) are ranked at the very top of the queue.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import duckdb
import pandas as pd
import os
from google.colab import userdata

# 1. Setup and Authenticate with Hugging Face
hf_token = userdata.get('HF_TOKEN')
conn = duckdb.connect()
conn.execute(f"""
    INSTALL httpfs;
    LOAD httpfs;
    CREATE SECRET (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

# The fact table containing our daily metrics
base_url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

# 2. Create the output directory to prevent path errors
os.makedirs('work/outputs', exist_ok=True)

# 3. Query the data and apply the Impressions vs Sessions rule
query_baseline = f"""
    WITH page_aggs AS (
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_impressions) as avg_impressions,
            AVG(ga4_sessions) as avg_sessions
        FROM '{base_url}/month=2026-03/*.parquet'
        GROUP BY 1, 2
    )
    SELECT
        client_hash_id,
        content_hash_id,
        avg_impressions,
        avg_sessions,
        avg_impressions AS action_score,
        'HIGH_IMPRESSION_LOW_SESSION' AS reason_code,
        'CTR_OPTIMIZATION_REVIEW' AS action_label
    FROM page_aggs
    WHERE avg_impressions > 100 AND avg_sessions < 2
    ORDER BY action_score DESC
"""

df_baseline = conn.execute(query_baseline).df()

# 4. Write to CSV
df_baseline.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Rule applied successfully.")
print(f"Generated a ranked queue of {len(df_baseline)} pages.")
print("Saved to 'work/outputs/baseline_action_score.csv'.\n")
print("Top 3 rows:")
print(df_baseline[['content_hash_id', 'action_score', 'avg_sessions', 'reason_code']].head(3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rule applied successfully.
Generated a ranked queue of 12235 pages.
Saved to 'work/outputs/baseline_action_score.csv'.

Top 3 rows:
            content_hash_id  action_score  avg_sessions  \
0  content_44f34c0a90047651   6851.741935      1.193548   
1  content_82e35c4845e6c391   4642.161290      1.903226   
2  content_8e1334d6356668e3   4354.322581      0.363636   

                   reason_code  
0  HIGH_IMPRESSION_LOW_SESSION  
1  HIGH_IMPRESSION_LOW_SESSION  
2  HIGH_IMPRESSION_LOW_SESSION  


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
print("--- TOP 20 REVIEW ---")
top_20 = df_baseline.head(20)

for i, row in top_20.iterrows():
    print(f"Rank {i+1} | Content: {row['content_hash_id'][:8]}...")
    print(f"  - Action: {row['action_label']} | Reason: {row['reason_code']}")
    print(f"  - Confidence note: Massive visibility ({row['action_score']:.0f} impressions/day) but abysmal traffic ({row['avg_sessions']:.1f} sessions/day). Prime target for title/meta rewrite.")
    print(f"  - What would make it wrong: If the GA4 tracking pixel broke on this specific page during March, the '0 sessions' is a reporting glitch, not an actual lack of traffic. An editor's time would be wasted.")
    print("")

--- TOP 20 REVIEW ---
Rank 1 | Content: content_...
  - Action: CTR_OPTIMIZATION_REVIEW | Reason: HIGH_IMPRESSION_LOW_SESSION
  - Confidence note: Massive visibility (6852 impressions/day) but abysmal traffic (1.2 sessions/day). Prime target for title/meta rewrite.
  - What would make it wrong: If the GA4 tracking pixel broke on this specific page during March, the '0 sessions' is a reporting glitch, not an actual lack of traffic. An editor's time would be wasted.

Rank 2 | Content: content_...
  - Action: CTR_OPTIMIZATION_REVIEW | Reason: HIGH_IMPRESSION_LOW_SESSION
  - Confidence note: Massive visibility (4642 impressions/day) but abysmal traffic (1.9 sessions/day). Prime target for title/meta rewrite.
  - What would make it wrong: If the GA4 tracking pixel broke on this specific page during March, the '0 sessions' is a reporting glitch, not an actual lack of traffic. An editor's time would be wasted.

Rank 3 | Content: content_...
  - Action: CTR_OPTIMIZATION_REVIEW | Reason: HIGH_I

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks (Why the baseline fails):
A fixed rule creates arbitrary cliffs. If a page is 179 days old and getting 10,000 impressions a day while actively plummeting in rank, our rule will completely ignore it because it missed the >180 cutoff by one day. A machine learning model will easily beat this because it weighs non-linear combinations of age and volume instead of using rigid IF/ELSE statements.

Leakage Check:
I can confirm there is no leakage in this baseline.

I only queried historical observable facts (content_age_days and gsc_impressions).

I deliberately did not use trend_direction (the target proxy) to build the rule.

I did not include any internal product flags like health_score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.